# 충남대 캠퍼스 챗봇 — Colab 데모 (7.8B + CPU 오프로드 + 안 끊기는 프록시)

새 Colab에서 위에서부터 셀을 순서대로 실행하세요. 런타임은 **GPU(T4)** 로 설정.
(메뉴: 런타임 > 런타임 유형 변경 > 하드웨어 가속기 = T4 GPU)

**핵심 설계**
- 생성모델 7.8B(4bit) + bge-m3 임베더/리랭커는 **CPU로 내려 VRAM 확보** → T4에서 OOM 회피
- **cloudflared 안 씀** → Colab 자체 포트 프록시로 노출(100초 한도 없음 = 524 없음)
- 느린 학과서버 크롤 타임아웃 12초 → 라이브 답이 100초 안에

속도 우선이면 4번 셀의 `MODEL_PRIMARY_NAME` 을 `...-2.4B-Instruct` 로 바꾸세요(더 빠르고 더 안전).

## 1. 코드 가져오기 (GitHub clone / pull)

In [ ]:
import os, subprocess
PROJ = '/content/cnu-llm-bot'
REPO = 'https://github.com/Longarden/cnu-llm-bot.git'
if not os.path.isdir(PROJ):
    subprocess.run(['git', 'clone', REPO, PROJ], check=True)
else:
    subprocess.run(['git', '-C', PROJ, 'pull'], check=False)
os.chdir(PROJ)
print('cwd =', os.getcwd())

## 2. 의존성 설치
torch 2.5.1 핀이라 설치 후 **런타임 재시작 안내가 뜨면 한 번 재시작**하고, 이 셀은 건너뛰고 3번부터 다시 실행하세요.

In [ ]:
!pip install -q -r requirements.txt
print('의존성 설치 완료')

## 3. 데이터 자동 준비 (model/ 분류기 + chroma_db/ 벡터DB)
이 둘은 용량 때문에 깃에 없음. 아래 셀이 **자동으로**:
1. 이미 있으면 → 그대로 사용
2. 드라이브 어딘가에 있으면 → 자동 탐색해서 복원 (경로 직접 안 박아도 됨)
3. 둘 다 없으면 → 깃에 든 데이터로 **새로 빌드** 후 드라이브(`MyDrive/cnu_assets`)에 저장 → **다음 실행부터는 즉시 복원**

(첫 빌드는 분류기 수분 / 벡터DB 10~20분. 한 번만 빌드하면 다음부터 안 함.)

In [ ]:
import os, sys, glob, tarfile, shutil, subprocess
from google.colab import drive
drive.mount('/content/drive')

DRIVE = '/content/drive/MyDrive'
CACHE = f'{DRIVE}/cnu_assets'      # 빌드 결과 캐시(다음 실행 때 여기서 자동 복원)
os.makedirs(CACHE, exist_ok=True)

def has_real(name):
    # git clone 은 model/ config 만 받음 → 실제 알맹이(가중치/sqlite)로 판단.
    if name == 'model':     return bool(glob.glob('model/*.safetensors') + glob.glob('model/*.bin'))
    if name == 'chroma_db': return bool(glob.glob('chroma_db/*.sqlite3'))
    return False

def find_on_drive(name):
    # 드라이브 어디에 있든 자동 탐색: 1) name.tar.gz  2) 풀린 폴더
    tgz = glob.glob(f'{DRIVE}/**/{name}.tar.gz', recursive=True)
    if tgz:
        return ('tgz', max(tgz, key=os.path.getsize))
    if name == 'model':
        hits = glob.glob(f'{DRIVE}/**/model/*.safetensors', recursive=True) + glob.glob(f'{DRIVE}/**/model/*.bin', recursive=True)
    else:
        hits = glob.glob(f'{DRIVE}/**/chroma_db/*.sqlite3', recursive=True)
    return ('dir', os.path.dirname(hits[0])) if hits else (None, None)

def restore(name):
    print(f'[restore] 드라이브에서 {name} 탐색 중...')
    kind, path = find_on_drive(name)
    if kind == 'tgz':
        with tarfile.open(path) as t: t.extractall('.')
        print(f'[restore] {name} <- {path}'); return True
    if kind == 'dir':
        os.makedirs(name, exist_ok=True)
        for f in glob.glob(path + '/*'):
            dst = os.path.join(name, os.path.basename(f))
            if not os.path.exists(dst):
                (shutil.copytree if os.path.isdir(f) else shutil.copy2)(f, dst)
        print(f'[restore] {name} <- {path} (폴더 복사)'); return True
    return False

def cache_to_drive(name):
    out = f'{CACHE}/{name}.tar.gz'
    with tarfile.open(out, 'w:gz') as t: t.add(name)
    print(f'[cache] {name} → {out}  (다음 실행부터는 이걸로 즉시 복원)')

BUILD = {  # 드라이브에도 없을 때만: 깃에 든 data/ 로 새로 빌드
    'model':     [sys.executable, 'scripts/train_classifier.py'],   # 분류기 학습(data/cls/train.json)
    'chroma_db': [sys.executable, 'scripts/rebuild_index.py'],       # 벡터DB 빌드(data/crawled, bge-m3)
}

for name in ('model', 'chroma_db'):
    if has_real(name):
        print(f'[ok] {name} 이미 있음'); continue
    if restore(name) and has_real(name):
        continue
    # 있으면 복원 끝. 드라이브에도 없으면 → 빌드 후 드라이브에 캐시(다음부터 복원).
    print(f'[build] {name} 드라이브에 없음 → 새로 빌드(시간 좀 걸림: 분류기 수분 / 벡터DB 10~20분)...')
    try:
        subprocess.run(BUILD[name], check=True, cwd=os.getcwd())
        if has_real(name):
            cache_to_drive(name)
        else:
            print(f'[build] !!! {name} 빌드했는데 결과물이 없음 — 로그 확인')
    except Exception as e:
        print(f'[build] !!! {name} 빌드 실패: {e}')

print('---')
print('분류기 가중치:', [os.path.basename(f) for f in glob.glob('model/*.safetensors') + glob.glob('model/*.bin')] or 'X 없음 → 분류기 안 됨')
print('벡터DB:', f"있음 ({count_str})" if (count_str := str(len(glob.glob('chroma_db/*.sqlite3'))) + ' sqlite') and has_real('chroma_db') else 'X 없음 → 정적 RAG 안 됨')

## 4. 환경 설정 + UI 실행 + 안 끊기는 Colab 프록시
모델 로드(워밍업)에 약 1분 걸립니다. '서버 준비됨' 뜬 뒤 아래 iframe에 챗봇이 나옵니다.

In [ ]:
import os, sys, threading, time, socket
sys.path.insert(0, os.getcwd())

# ── 생성 모델: 품질 우선 7.8B(4bit). 속도/안전 우선이면 ...-2.4B-Instruct 로 변경 ──
os.environ['MODEL_PRIMARY_NAME'] = 'LGAI-EXAONE/EXAONE-3.5-7.8B-Instruct'
# ── 임베더 CPU 고정. 리랭커는 VRAM 남으면 'cuda'로 올리면 정적질문 2~4초 빨라짐 ──
os.environ['EMBED_DEVICE']  = 'cpu'
os.environ['RERANK_DEVICE'] = 'cpu'   # VRAM 4GB+ 남으면 'cuda' 로 바꿔도 됨
# ── 로컬 생성 + 라이브크롤 + 리랭커 ON ──
os.environ['GEN_BACKEND']   = 'local'
os.environ['CHAT_REALTIME'] = '1'
os.environ['RERANK']        = '1'
os.environ['GRADIO_SHARE']  = '0'   # cloudflared 미사용(100초 한도 회피)
# ── VRAM 단편화 방지 + 크롤 타임아웃((connect, read)) ──
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['CRAWL_CONNECT_TIMEOUT'] = '4'
os.environ['CRAWL_TIMEOUT'] = '15'
os.environ['CRAWL_RETRIES'] = '0'

from src.chatbot_ui import launch_app
threading.Thread(target=lambda: launch_app(share=False), daemon=True).start()

PORT = 7860
print('워밍업 대기중... (모델 로드 ~1분)')
for _ in range(420):
    try:
        socket.create_connection(('127.0.0.1', PORT), 0.5).close()
        print('서버 준비됨'); break
    except OSError:
        time.sleep(1)

# Colab 프록시가 만든 '진짜 접속 링크'를 문자열로 받아 출력(= https://localhost:7860/ 형태, 작동함).
# launch_app 이 찍는 http://localhost:7860 은 프록시 미경유라 Colab에선 안 열림 → 무시.
from google.colab.output import eval_js
from google.colab import output
url = eval_js(f'google.colab.kernel.proxyPort({PORT})')
print('\n================ 접속 링크 (클릭/복사) ================')
print('   ', url)
print('   (위 [ui] 로컬: http://localhost:7860 은 Colab에선 안 됨 — 무시)')
print('=====================================================\n')
output.serve_kernel_port_as_iframe(PORT, height='640')   # 셀 안 임베드

## 참고
- **GPU 확인:** 아래 셀에서 `!nvidia-smi` — python 프로세스가 **한 개만** 보여야 정상(여러 개면 이전 실행이 안 죽은 것 → 런타임 재시작).
- **안 끊김:** 위 iframe은 Colab 프록시라 cloudflared의 100초 한도가 없음 → 7.8B 느린 답도 524 안 남.
- **느리면:** 4번 셀 `MODEL_PRIMARY_NAME` 을 `LGAI-EXAONE/EXAONE-3.5-2.4B-Instruct` 로 바꾸면 답이 초 단위(채점은 '말 되면 통과'라 2.4B로 충분).
- **OOM 나면:** 런타임 재시작(이전 프로세스 제거) 후 4번 셀부터 다시. 그래도 빠듯하면 2.4B로.

In [ ]:
!nvidia-smi